<a href="https://colab.research.google.com/github/amruthenium/MATSimplex/blob/main/Network_TwISXMLtoRDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ERROR: Could not find a version that satisfies the requirement csv (from versions: none)
ERROR: No matching distribution found for csv
ERROR: Could not find a version that satisfies the requirement sys (from versions: none)
ERROR: No matching distribution found for sys
ERROR: Could not find a version that satisfies the requirement time (from versions: none)
ERROR: No matching distribution found for time


In [ ]:
import csv, sys, time
from lxml import etree
SRC = "/content/drive/MyDrive/Thesis_TUM/munich.output_network.xml"
OUT = "/content/networkfinal2_rdf_trip.ttl"

LIMIT = None
if len(sys.argv) > 1:
    try:
        LIMIT = int(sys.argv[1])
    except ValueError:
        print(f"Warning: Could not convert '{sys.argv[1]}' to an integer. LIMIT will remain None.")



In [ ]:
CRS ="http://www.opengis.net/def/crs/EPSG/0/31468"

In [ ]:
PREFIXES = f"""@prefix twis: <https://w3id.org/twis#> .
@prefix mun:  <https://example.org/matsim/munich/> .
@prefix geo:  <http://www.opengis.net/ont/geosparql#> .
@prefix sf:   <http://www.opengis.net/ont/sf#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .
"""

In [ ]:
def dbl(v): #typed double literal
    return f'"{v}"^^xsd:double'

t0 = time.time()
coords = {}
emitted = set()
n_nodes = n_links = n_trip = 0

out = open(OUT, "w")
out.write(PREFIXES) #script to write prefixes in rdf triple file


307

In [ ]:
def emit_node(nid):
  global n_nodes, n_trip
  if nid in emitted:
    return
  x,y = coords[nid]
  out.write(
      f"mun:nodes_{nid} a twis:Streetnode, geo:Feature; \n"
      f" geo:hasGeometry mun:geom_nodes_{nid} .\n"
      f"mun:geom_nodes_{nid} a sf:Point ;\n"
      f'geo:asWKT "<{CRS}> POINT({x} {y})"^^geo:wktLiteral .\n')
  emitted.add(nid); n_nodes += 1; n_trip += 5;

ctx = etree.iterparse(SRC, events=("end",), tag=("node", "link"))
for _, elem in ctx:
  if elem.tag == "node":
    coords[elem.get("id")] = (elem.get("x"), elem.get("y"))
  else:
    if LIMIT is not None and n_links >= LIMIT:
      elem.clear()
      break
    lid = elem.get("id"); frm = elem.get("from"); to = elem.get("to")
    at = {a.get("name"): a.text for a in elem.iter("attribute")}
    fx, fy = coords.get(frm, ("", "")); tx, ty = coords.get(to, ("",""))
    wkt = f"<{CRS}> LINESTRING({fx} {fy}, {tx} {ty})"
    out.write(
        f"mun:link_{lid} a twis:Link, geo:Feature ;\n"
        f"twis:length {dbl(elem.get('length'))} ;\n"
        f"twis:freespeed {dbl(elem.get('freespeed'))} ;\n"
        f"twis:permlanes {dbl(elem.get('permlanes'))} ;\n"
        f"twis:capacity {dbl(elem.get('capacity'))} ;\n"
        f"twis:oneway \"{elem.get('oneway')}\" ;\n"
        f"twis:modes \"{elem.get('modes')}\" ;\n"
        f"twis:classification \"{at.get('type','')}\" ;\n"
        f"twis:origid \"{at.get('origid','')}\" ;\n"
        f"twis:fromNode mun:nodes_{frm} ;\n"
        f"twis:toNode mun:nodes_{to} ;\n"
        f"geo:hasGeometry mun:geom_link_{lid} .\n"
        f"mun:geom_link_{lid} a sf:LineString ;\n"
        f'geo:asWKT "{wkt}"^^geo:wktLiteral .\n'
        f"mun:linkPort_{lid}_A a twis:LinkPort ;\n"
        f' twis:Status "Start"; twis:link mun:link_{lid} ; twis:node mun:nodes_{frm} .\n'
        f"mun:linkPort_{lid}_E a twis:LinkPort ;\n"
        f' twis:Status "End"; twis:link mun:link_{lid} ; twis:node mun:nodes_{to} .\n')
    n_links += 1; n_trip += 23 #number needs to be a value of total objects and classes, needs to be more robust and generalisable.
    emit_node(frm); emit_node(to)
    elem.clear()
    while elem.getprevious() is not None:
        del elem.getparent()[0]
out.close()


In [ ]:
print(f"wrote {OUT}")
print(f"Ast (links): {n_links:,}   Strassenknotenpunkt (nodes): {n_nodes:,}")
print(f"~triples written: {n_trip:,}   elapsed {time.time()-t0:.1f}s")
proj = 212772*5 + 499435*23 #number needs to be a value of total objects and classes, needs to be more robust and generalisable.
print(f"projected full-network triples: ~{proj:,}")

wrote /content/networkfinal2_rdf_trip.ttl
Ast (links): 499,435   Strassenknotenpunkt (nodes): 212,772
~triples written: 12,550,865   elapsed 54.5s
projected full-network triples: ~12,550,865


In [ ]:
!pip install rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 10.1 MB/s eta 0:00:00


In [ ]:
#sample SPARQL queries
#PREFIX twis: <https://w3id.org/twis#>
#SELECT ?classification (COUNT(?link) AS ?n) WHERE {
  #?link a twis:Link ; twis:classification ?classification .
#} GROUP BY ?classification ORDER BY DESC(?n)

In [ ]:
import rdflib

# Create a graph and parse the Turtle file
g = rdflib.Graph()
g.parse('networksample_rdf_trip.ttl', format='turtle')

# View the contents by printing the graph or iterating through statements
print(g)
for stmt in g:
    print(stmt)